# Lab Experiment: Data Collection
## Web Scraping and APIs

**Course:** Data science  Laboratory
**Class:** B.Tech
**Duration:** 2 hours

---

### Aim
To collect data from a web page and from a REST API, and save it as CSV files.

### Objectives
1. Extract data from an HTML page using **BeautifulSoup**
2. Extract a table using **pandas.read_html**
3. Collect data from a **REST API** and parse the JSON response
4. Save all collected data to CSV files

### Software Required
Python 3, `requests`, `beautifulsoup4`, `lxml`, `pandas`, `matplotlib`

### Theory

There are two ways to collect data from the internet:

| | Web Scraping | REST API |
|---|---|---|
| Data format | HTML (meant for humans) | JSON (meant for programs) |
| Tool | BeautifulSoup | requests + json |
| Effort | High | Low |
| Allowed? | Only if the site permits it | Yes, that is what it is for |

**Rule: if an API exists, use the API.** Scrape only when there is no API.

---
## Step 0: Setup

In [2]:
# Install if needed (uncomment the line below)
# !pip install requests beautifulsoup4 lxml pandas matplotlib

import io
import json
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from pathlib import Path

# Folder to store collected data
DATA = Path("data")
DATA.mkdir(exist_ok=True)

print("Setup complete. Data will be saved in:", DATA.resolve())

Setup complete. Data will be saved in: /Users/sahilkhan/Data science/data


---
## Step 1: Rules Before Collecting Data

Before scraping any website, check these four things:

1. **Check `robots.txt`** — visit `https://website.com/robots.txt`. It says which pages
   crawlers may visit.
2. **Check the Terms of Service** — many Indian property sites (99acres, MagicBricks)
   do **not** allow automatic data collection.
3. **Go slow** — wait 1–2 seconds between requests. Do not send requests in a fast loop.
4. **Never collect personal data** — no names, phone numbers or email IDs.

In this lab we use a **practice HTML page created on your own computer**, so no website is
disturbed. The API we use (Open-Meteo) is free and openly allows this.

In [3]:
# Check if a website allows scraping
from urllib.robotparser import RobotFileParser

def can_scrape(url):
    rp = RobotFileParser()
    rp.set_url(url.split("/")[0] + "//" + url.split("/")[2] + "/robots.txt")
    try:
        rp.read()
        return rp.can_fetch("*", url)
    except Exception:
        return "could not check (no internet)"

print("Wikipedia :", can_scrape("https://en.wikipedia.org/wiki/Bangalore"))
print("99acres   :", can_scrape("https://www.99acres.com/property-in-bangalore-ffid"))

Wikipedia : False
99acres   : False


---
# EXPERIMENT 1
## Collecting Data from an HTML Page

We will collect Bengaluru house listings from a web page.

**How scraping works:**

```
HTML page  →  BeautifulSoup  →  find the tags  →  get the text  →  DataFrame  →  CSV
```

[![Chat-GPT-Image-Aug-13-2026-06-43-45-PM.png](https://i.postimg.cc/NjJc6QCL/Chat-GPT-Image-Aug-13-2026-06-43-45-PM.png)](https://postimg.cc/JskvmCLW)

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. URL
url = "https://quotes.toscrape.com/"

# 2. Fetch
response = requests.get(url)

# 3. Parse
soup = BeautifulSoup(response.text, "html.parser")

# 4. Find quote containers
quotes = soup.find_all("div", class_="quote")  # ← fix: div not span

# 5. Extract
data = []
for quote in quotes:
    text   = quote.find("span", class_="text").get_text(strip=True)
    author = quote.find("small", class_="author").get_text(strip=True)
    data.append({"Quote": text, "Author": author})

# 6. DataFrame
df = pd.DataFrame(data)
print(df)

# 7. Save
df.to_csv("quotes.csv", index=False)
print("Data saved to quotes.csv")

                                               Quote             Author
0  “The world as we have created it is a process ...    Albert Einstein
1  “It is our choices, Harry, that show what we t...       J.K. Rowling
2  “There are only two ways to live your life. On...    Albert Einstein
3  “The person, be it gentleman or lady, who has ...        Jane Austen
4  “Imperfection is beauty, madness is genius and...     Marilyn Monroe
5  “Try not to become a man of success. Rather be...    Albert Einstein
6  “It is better to be hated for what you are tha...         André Gide
7  “I have not failed. I've just found 10,000 way...   Thomas A. Edison
8  “A woman is like a tea bag; you never know how...  Eleanor Roosevelt
9  “A day without sunshine is like, you know, nig...       Steve Martin
Data saved to quotes.csv


In [5]:
import json
import re
import pandas as pd
 
INPUT_FILE = "data/House.html"
OUTPUT_FILE = "bengaluru_houses.csv"
 
 
# ---------- STEP 1: read the file ----------
html = open(INPUT_FILE, encoding="utf-8").read()
print("File read. Size:", len(html), "characters")
 
 
# ---------- STEP 2: cut out the JSON ----------
marker = "window.__initialData__="
start = html.find(marker) + len(marker)
 

File read. Size: 1774012 characters


In [6]:
import requests

url = "https://www.99acres.com/property-in-bangalore-ffid"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
html = response.text
print("Page fetched. Size:", len(html), "chars")

# Find the actual JSON data marker
marker = "window.__INITIAL_STATE__ ="
marker_pos = html.find(marker)

if marker_pos == -1:
    raise ValueError("Marker not found — inspect page source to find the right marker")

start = html.index("{", marker_pos)
print("JSON start found at index:", start)

Page fetched. Size: 405 chars


ValueError: Marker not found — inspect page source to find the right marker

In [ ]:
html = open("data/House.html", encoding="utf-8").read()
start = html.index("{", html.find("window.__initialData__=") + len("window.__initialData__="))

In [ ]:

 
# Read forward until the opening { is closed by its matching }
depth = 0
for end in range(start, len(html)):
    if html[end] == "{":
        depth += 1
    elif html[end] == "{":
        depth -= 1
        if depth == 0:
            break
 

In [ ]:
start = html.find("{")

depth = 0

for end in range(start, len(html)):
    if html[end] == "{":
        depth += 1
    elif html[end] == "}":
        depth -= 1

        if depth == 0:
            break

import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://quotes.toscrape.com/"
r = requests.get(url)

html = r.text

# Find the quotes
soup = BeautifulSoup(html, "html.parser")

quotes = []

for quote in soup.select(".quote"):
    text = quote.select_one(".text").get_text(strip=True)
    author = quote.select_one(".author").get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in quote.select(".tag")]

    quotes.append({
        "quote": text,
        "author": author,
        "tags": ", ".join(tags)
    })

# Convert to DataFrame
df = pd.DataFrame(quotes)

# Save CSV
df.to_csv("quotes.csv", index=False)

print("Data saved to quotes.csv")
print(df.head())

In [7]:
import json
import pandas as pd

OUTPUT_FILE = "99acres_bangalore.csv"

# STEP 1: Read the saved HTML file
with open("data/House.html", "r", encoding="utf-8") as f:
    html = f.read()

print("HTML loaded successfully")
print("HTML length:", len(html))

# STEP 2: Extract __initialData__ JSON
marker = "window.__initialData__="

start = html.find(marker)

if start == -1:
    raise ValueError("Could not find window.__initialData__ in House.html")

start += len(marker)

# Use JSONDecoder to read the complete JSON object
decoder = json.JSONDecoder()
data, end = decoder.raw_decode(html[start:])

print("JSON loaded successfully")

# STEP 3: Reach the list of properties
properties = data["srp"]["pageData"]["properties"]

print("Number of listings found:", len(properties))

# STEP 4: Pick the fields we want

def price_in_lakh(rupees):
    try:
        return round(float(rupees) / 100000, 2)
    except (TypeError, ValueError):
        return None


def to_number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


rows = []

for p in properties:
    rows.append({
        "location": p.get("LOCALITY"),
        "bhk": to_number(p.get("BEDROOM_NUM")),
        "sqft": to_number(p.get("LOCALIZED_AREA_VALUE")),
        "area_unit": p.get("LOCALIZED_AREA_UNIT_LABEL"),
        "price_lakh": price_in_lakh(p.get("MIN_PRICE")),
        "price_text": p.get("PRICE"),
        "price_per_sqft": to_number(p.get("PRICE_SQFT")),
        "property_name": p.get("PROP_NAME"),
    })

# STEP 5: Create DataFrame
houses = pd.DataFrame(rows)

print("\nAll areas are sqft?")
print((houses["area_unit"] == "sqft").all())

print("\nMissing values:")
print(houses.isna().sum())

# STEP 6: Save CSV
houses.to_csv(OUTPUT_FILE, index=False)

print("\nSaved", len(houses), "rows to", OUTPUT_FILE)

# Show first 5 rows
print(houses.head().to_string(index=False))

HTML loaded successfully
HTML length: 1774012
JSON loaded successfully
Number of listings found: 27

All areas are sqft?
True

Missing values:
location          0
bhk               0
sqft              0
area_unit         0
price_lakh        0
price_text        0
price_per_sqft    0
property_name     0
dtype: int64

Saved 27 rows to 99acres_bangalore.csv
                        location  bhk   sqft area_unit  price_lakh      price_text  price_per_sqft              property_name
          Bommasandra, Bangalore  5.0 2600.0      sqft      219.99          2.2 Cr          8461.0                SKR Gardens
5th Block Hbr Layout, HBR Layout 12.0 1200.0      sqft      380.00          3.8 Cr         31666.0                           
        Sarjapur Road, Bangalore  4.0 3040.0      sqft      225.00 2.25  - 3.28 Cr          9087.0 CasaLife by Bhavisha Homes
           Whitefield, Bangalore  5.0 4629.0      sqft      381.38 3.81  - 6.65 Cr         11299.0                 DSR Elixir
           Whi

##### Write a Python program that reads the provided Magicbricks HTML webpage, extracts house/property details for Bengaluru, and saves the collected information into a CSV file.

### Result — Experiment 1

**Number of listings collected:** ______

**Columns collected:** ______

**File saved:** ______

**Observation:**

> ...